# Fleet Capacity-Tightness Frontier

This notebook replaces the old fleet-size heatmap. It fixes the fleet size at `N=20` and varies station-capacity tightness around an instance-specific robust feasibility threshold.

For each ACN-derived fleet instance, it first computes

\[
M_{\mathrm{rob,min}} = \min M \quad \text{such that the robust fleet model is feasible.}
\]

It then evaluates deterministic, Slotwise Q75, and robust protection at

\[
M(\gamma) = (1+\gamma)M_{\mathrm{rob,min}}.
\]

This gives a cleaner SEGAN-style experiment: as the station or transformer capacity margin becomes tighter, how do cost, V2G operation, reliability, and complementarity diagnostics change?

In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import time

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import ev_robust_charging as ev

## User-tunable configuration

Change only this cell for most sensitivity checks. `CAPACITY_GAMMAS` controls how loose the station capacity is relative to the robust-feasible minimum for each seed. `0.00` is the tightest robust-feasible case.

In [ ]:
N_VEHICLES = 20
DELTA = 0.5
CAPACITY_GAMMAS = (0.00, 0.02, 0.05, 0.10, 0.15)
FLEET_SEEDS = ev.FLEET_SEEDS
POLICIES = ev.POLICIES

# Numerical guard added after M(gamma). Set to 0.0 for the exact formula in the paper.
CAPACITY_BUFFER = 1.0e-4

# The row helper in ev_robust_charging uses 10,000 Monte Carlo samples internally.
# Keep the note here so the protocol is visible in the notebook.
MC_SAMPLES_FOR_EVALUATION = 10_000

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

POLICY_ORDER = [policy.label for policy in POLICIES]
POLICY_COLORS = {
    "Deterministic": "#8C2D04",
    "Slotwise Q75": "#2B5C8A",
    "Robust": "#1B7837",
}

print(f"Capacity-tightness configuration: N={N_VEHICLES}, delta={DELTA}, gammas={CAPACITY_GAMMAS}")
print(f"Seeds: {len(FLEET_SEEDS)}")
print(f"Outputs: {OUT_DIR}")

## Load base profile and ACN-derived fleet inputs

The fleet instances use the same OPSD base day and ACN-Data construction as the main paper. The capacity limit is not fixed here; each seed gets its own `M_rob_min` from the robust feasibility subproblem.

In [ ]:
base = ev.build_base_case()
fleet_config = ev.FleetExperimentConfig()
acn_cache = ev.acn_cache_path(
    fleet_config.acn_site,
    fleet_config.acn_start,
    fleet_config.acn_end,
    fleet_config.acn_cache,
)
acn_sessions = ev.load_acn_sessions(
    fleet_config.acn_site,
    fleet_config.acn_start,
    fleet_config.acn_end,
    fleet_config.acn_min_kwh,
    acn_cache,
)
horizon_dates = ev.feasible_acn_horizon_dates(
    acn_sessions,
    base=base,
    site=fleet_config.acn_site,
    slot_minutes=fleet_config.slot_minutes,
    min_sessions=N_VEHICLES,
)

metadata = pd.DataFrame([
    {
        "fleet_size": N_VEHICLES,
        "delta": DELTA,
        "capacity_gammas": ",".join(f"{g:g}" for g in CAPACITY_GAMMAS),
        "num_seeds": len(FLEET_SEEDS),
        "acn_site": fleet_config.acn_site,
        "acn_start": fleet_config.acn_start,
        "acn_end": fleet_config.acn_end,
        "raw_sessions_after_filter": len(acn_sessions),
        "valid_horizon_days": len(horizon_dates),
        "mc_samples_for_evaluation": MC_SAMPLES_FOR_EVALUATION,
    }
])
metadata.to_csv(OUT_DIR / "capacity_tightness_metadata.csv", index=False)
pd.DataFrame({"horizon_date": horizon_dates}).to_csv(OUT_DIR / "capacity_tightness_horizon_dates.csv", index=False)

print(metadata.to_string(index=False))
print(f"First horizon dates: {horizon_dates[:5]}")

## Helper functions

`robust_min_capacity` solves one convex capacity-sizing problem for each seed. It is separate from the policy evaluation problems. The evaluated policies are then solved at `M(gamma)`.

In [ ]:
OPTIMAL_STATUSES = {cp.OPTIMAL, cp.OPTIMAL_INACCURATE}


def build_seed_case(seed: int) -> ev.FleetData:
    return ev.build_fleet_case_for_experiment(
        base,
        n_vehicles=N_VEHICLES,
        seed=seed,
        config=fleet_config,
        acn_sessions=acn_sessions,
        horizon_dates=horizon_dates,
    )


def robust_min_capacity(data: ev.FleetData, delta: float) -> dict:
    # Return the smallest station-capacity limit that keeps the robust fleet model feasible.
    _, upper = ev.fleet_uncertainty_box(data, delta)

    x = cp.Variable((data.N, data.T), nonneg=True, name="min_capacity_charge")
    y = cp.Variable((data.N, data.T), nonneg=True, name="min_capacity_discharge")
    E = cp.Variable((data.N, data.T + 1), name="min_capacity_soc")
    m_cap = cp.Variable(name="robust_min_capacity")

    aggregate_charge = cp.sum(x, axis=0)
    aggregate_discharge = cp.sum(y, axis=0)
    station_load = data.L + aggregate_charge - aggregate_discharge

    constraints = [
        x <= data.X.reshape(data.N, 1) * data.availability,
        y <= data.Y.reshape(data.N, 1) * data.availability,
        E[:, 0] == data.E0,
        E[:, 1:] == E[:, :-1]
        + cp.multiply(data.eta_c.reshape(data.N, 1), x)
        - cp.multiply((1.0 / data.eta_d).reshape(data.N, 1), y),
        E[:, 1:] >= data.Emin.reshape(data.N, 1),
        E[:, 1:] <= data.Emax.reshape(data.N, 1),
        E[:, -1] >= data.Etarget,
        data.L + cp.sum(cp.multiply(upper, x), axis=0) <= m_cap,
        m_cap >= float(np.max(data.L)),
    ]

    # Tiny secondary terms pick a stable feasible schedule without changing the capacity scale.
    secondary = 1.0e-9 * (cp.sum(x + y) + cp.sum_squares(station_load) / max(1.0, data.T))
    problem = cp.Problem(cp.Minimize(m_cap + secondary), constraints)
    start = time.perf_counter()
    ev.solve_fleet_with_fallback(problem)
    elapsed = time.perf_counter() - start

    result = {
        "status": problem.status,
        "M_rob_min": np.nan,
        "capacity_solve_time_sec": elapsed,
        "capacity_problem_value": np.nan,
    }
    if problem.status in OPTIMAL_STATUSES:
        result["M_rob_min"] = float(m_cap.value)
        result["capacity_problem_value"] = float(problem.value)
    return result


def evaluate_policy_at_capacity(
    data0: ev.FleetData,
    seed: int,
    M_rob_min: float,
    gamma: float,
    policy: ev.Policy,
) -> dict:
    capacity = float((1.0 + gamma) * M_rob_min + CAPACITY_BUFFER)
    data = replace(data0, M=capacity)
    solution = ev.solve_fleet_model(data, delta=DELTA, policy=policy)
    solution["seed"] = seed
    row = ev.row_from_fleet_solution(data, solution, delta=DELTA)
    row.update(
        {
            "capacity_gamma": gamma,
            "capacity_ratio": capacity / M_rob_min,
            "M_rob_min": M_rob_min,
            "capacity_buffer": CAPACITY_BUFFER,
        }
    )
    return row

## Run capacity-tightness experiment

This is the only long-running cell. It prints feasibility counts so you can immediately see whether any seed/policy/capacity combination failed.

In [ ]:
capacity_rows = []
raw_rows = []

for seed_idx, seed in enumerate(FLEET_SEEDS, start=1):
    data0 = build_seed_case(seed)
    cap = robust_min_capacity(data0, DELTA)
    capacity_rows.append(
        {
            "seed": seed,
            "seed_index": seed_idx,
            "horizon_date": data0.horizon_date,
            "n_vehicles": data0.N,
            "source_sessions": data0.source_sessions,
            **cap,
        }
    )

    if not np.isfinite(cap["M_rob_min"]):
        print(f"[{seed_idx:02d}/{len(FLEET_SEEDS)}] seed={seed}: capacity sizing failed, status={cap['status']}")
        continue

    for gamma in CAPACITY_GAMMAS:
        for policy in POLICIES:
            row = evaluate_policy_at_capacity(data0, seed, cap["M_rob_min"], gamma, policy)
            row["seed_index"] = seed_idx
            raw_rows.append(row)

    solved = len(CAPACITY_GAMMAS) * len(POLICIES)
    print(
        f"[{seed_idx:02d}/{len(FLEET_SEEDS)}] seed={seed}: "
        f"M_rob_min={cap['M_rob_min']:.3f}, evaluated {solved} policy-capacity cases"
    )

capacity_df = pd.DataFrame(capacity_rows)
raw_df = pd.DataFrame(raw_rows)

capacity_df.to_csv(OUT_DIR / "fleet_capacity_tightness_min_capacity.csv", index=False)
raw_df.to_csv(OUT_DIR / "fleet_capacity_tightness_raw.csv", index=False)

print("\nCapacity sizing statuses:")
print(capacity_df["status"].value_counts(dropna=False).to_string())

print("\nPolicy evaluation statuses:")
print(raw_df.groupby(["capacity_gamma", "policy_label", "status"]).size().rename("count").reset_index().to_string(index=False))

optimal_count = int(raw_df["status"].isin(OPTIMAL_STATUSES).sum()) if not raw_df.empty else 0
print(f"\nOptimal policy evaluations: {optimal_count}/{len(raw_df)}")

## Summaries for paper tables

The main table is grouped by capacity slack and policy. `WC excess` is the worst-case station-capacity excess under the box uncertainty set, truncated at zero only for the table column `mean_positive_wc_excess`.

In [ ]:
if raw_df.empty:
    raise RuntimeError("No policy evaluations were produced. Run the previous cell first.")

work = raw_df.copy()
work["is_optimal"] = work["status"].isin(OPTIMAL_STATUSES)
work["reported_violation"] = work["violation_probability"]
work["wc_excess"] = work["worst_case_margin"]
work["positive_wc_excess"] = work["wc_excess"].clip(lower=0.0)
work["comp_residual"] = work["max_simultaneous_flow"]

summary = (
    work.groupby(["capacity_gamma", "policy_label"], dropna=False)
    .agg(
        runs=("status", "size"),
        mean_capacity_ratio=("capacity_ratio", "mean"),
        optimal_runs=("is_optimal", "sum"),
        mean_M=("grid_limit", "mean"),
        mean_M_rob_min=("M_rob_min", "mean"),
        mean_eur_eq_cost=("economic_cost", "mean"),
        mean_v2g_kwh=("total_discharge", "mean"),
        mean_violation=("reported_violation", "mean"),
        max_violation=("reported_violation", "max"),
        mean_wc_excess=("wc_excess", "mean"),
        mean_positive_wc_excess=("positive_wc_excess", "mean"),
        max_wc_excess=("wc_excess", "max"),
        mean_solve_time_sec=("solve_time_sec", "mean"),
        max_comp_residual=("comp_residual", "max"),
    )
    .reset_index()
)
summary["capacity_gamma"] = summary["capacity_gamma"].astype(float)
summary["policy_label"] = pd.Categorical(summary["policy_label"], categories=POLICY_ORDER, ordered=True)
summary = summary.sort_values(["capacity_gamma", "policy_label"]).reset_index(drop=True)
summary.to_csv(OUT_DIR / "table8_capacity_tightness_frontier.csv", index=False)

compact = summary[[
    "capacity_gamma",
    "policy_label",
    "optimal_runs",
    "runs",
    "mean_eur_eq_cost",
    "mean_v2g_kwh",
    "mean_violation",
    "max_violation",
    "mean_positive_wc_excess",
    "mean_solve_time_sec",
    "max_comp_residual",
]].copy()
compact.to_csv(OUT_DIR / "table8_capacity_tightness_compact.csv", index=False)

print(compact.to_string(index=False, float_format=lambda x: f"{x:.4g}"))

## Figure 3: capacity-tightness frontier

This figure reports the fleet capacity-tightness frontier. Smaller `gamma` means tighter station capacity.

In [ ]:
def save_show(fig: plt.Figure, name: str, width: int = 1200) -> None:
    png = OUT_DIR / f"{name}.png"
    pdf = OUT_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=220, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved {png} and {pdf}")
    plt.close(fig)
    display(Image(filename=str(png), width=width))


plot_df = summary.copy()
plot_df["capacity_gamma_pct"] = 100.0 * plot_df["capacity_gamma"]

# Compute cost premium relative to Deterministic at the same gamma
det_cost = (
    plot_df[plot_df["policy_label"] == "Deterministic"]
    [["capacity_gamma", "mean_eur_eq_cost"]]
    .rename(columns={"mean_eur_eq_cost": "det_cost"})
)

plot_df = plot_df.merge(det_cost, on="capacity_gamma", how="left")
plot_df["cost_premium_vs_det"] = plot_df["mean_eur_eq_cost"] - plot_df["det_cost"]

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10.5,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), constrained_layout=True)

# Panel A: reliability
for policy in POLICY_ORDER:
    frame = plot_df[plot_df["policy_label"] == policy].sort_values("capacity_gamma")
    color = POLICY_COLORS.get(policy)

    axes[0].plot(
        frame["capacity_gamma_pct"],
        frame["mean_violation"],
        marker="o",
        linewidth=2.4,
        label=policy,
        color=color,
    )

axes[0].set_title("A. Reliability")
axes[0].set_xlabel(r"Capacity margin $\gamma$ (%)")
axes[0].set_ylabel("Mean violation probability")
axes[0].set_ylim(bottom=0.0)
axes[0].grid(alpha=0.25)

# Panel B: cost premium relative to Deterministic
# Deterministic is the zero baseline; plot it as a horizontal reference line.
axes[1].axhline(0.0, linewidth=1.2, linestyle="--", color=POLICY_COLORS.get("Deterministic"))

for policy in POLICY_ORDER:
    if policy == "Deterministic":
        continue

    frame = plot_df[plot_df["policy_label"] == policy].sort_values("capacity_gamma")
    color = POLICY_COLORS.get(policy)

    axes[1].plot(
        frame["capacity_gamma_pct"],
        frame["cost_premium_vs_det"],
        marker="o",
        linewidth=2.4,
        label=policy,
        color=color,
    )

axes[1].set_title("B. Cost premium")
axes[1].set_xlabel(r"Capacity margin $\gamma$ (%)")
axes[1].set_ylabel("Cost premium relative to Deterministic")
axes[1].set_ylim(bottom=0.0)
axes[1].grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 1.08),
)

save_show(fig, "figure3_capacity_tightness_frontier")


## Optional: feasibility and solver diagnostics figure

Use this only if you want a supplemental numerical diagnostics plot. The paper can keep Table 8 instead.

In [ ]:
diag = summary.copy()
diag["capacity_gamma_pct"] = 100.0 * diag["capacity_gamma"]
diag["optimal_fraction"] = diag["optimal_runs"] / diag["runs"]

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
for policy in POLICY_ORDER:
    frame = diag[diag["policy_label"] == policy].sort_values("capacity_gamma")
    color = POLICY_COLORS.get(policy)
    axes[0].plot(frame["capacity_gamma_pct"], frame["optimal_fraction"], marker="o", linewidth=2.2, label=policy, color=color)
    axes[1].plot(frame["capacity_gamma_pct"], frame["mean_solve_time_sec"], marker="o", linewidth=2.2, label=policy, color=color)

axes[0].set_title("Feasible/optimal fraction")
axes[0].set_xlabel(r"Capacity margin $\gamma$ (%)")
axes[0].set_ylabel("Optimal runs / total runs")
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.25)

axes[1].set_title("Solver time")
axes[1].set_xlabel(r"Capacity margin $\gamma$ (%)")
axes[1].set_ylabel("Mean solve time (s)")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

save_show(fig, "capacity_tightness_solver_diagnostics", width=1000)


## Suggested paper text

Use this experiment as the fleet-level replacement:

- Table 8: `table8_capacity_tightness_compact.csv` or the full `table8_capacity_tightness_frontier.csv`.
- Figure 3: `figure3_capacity_tightness_frontier.*`.

Suggested caption:

> Fleet-level capacity-tightness frontier for ACN-derived fleets with `N=20`. For each seed, `M_rob,min` is the smallest station-capacity limit under which the robust protected fleet model remains feasible. The evaluated capacity is `M(gamma)=(1+gamma)M_rob,min`. Smaller `gamma` therefore represents a tighter station or transformer capacity margin.